# Kinematic-features FLAMINGO

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Corrected TNG300 comparisons

Restart the kernel and run from the top. Set `LG_PROJECT` to your project directory on Roihu before running the setup cell. This notebook reads the corrected selection and existing feature caches; it does not require newly trained NPE checkpoints.

Inputs are verified against the v3 manifests. The corrected TNG sample has 1,530 pairs: 1,253 same-host and 277 different-host. Figures are written to `pickles_v3/figures/Kinematic-features_FLAMINGO`. Selection and feature caches are read only. Both simulations' radial pair velocities are **peculiar**; these comparison plots do not add a Hubble term.

Pair distributions count one row per pair. Galaxy-feature distributions explicitly distinguish unique final-pair members from pair-member entries (a galaxy may appear repeatedly). KS/energy/Wasserstein distances are descriptive; shared galaxies and hosts invalidate naive independent-row significance claims. Historical interpretation notes below require reassessment after rerunning.

Raw SOAP metadata, raw TNG catalogue checks, and large parent-catalogue diagnostics are optional flags in the setup cell and are off by default. Existing originals and outputs are preserved in the dated comparison-update backup.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from comparison_utils import (load_comparison_inputs, finite_lookup,
    normalized_wasserstein, group_blocks, resample_blocks)
from splits import pair_groups, group_split
RESULTS_DIR, PLOT_DIR, tng_dir, TNG_DATA, TNG_MANIFEST = load_comparison_inputs(
    PROJECT, 'Kinematic-features_FLAMINGO')
RUN_SOAP_METADATA = False  # Optional remote metadata lookup; requires hdfstream.
RUN_TNG_CATALOGUE_CHECKS = False  # Optional raw catalogue; set TNG_BASE when enabled.
RUN_PARENT_CATALOGUE_CHECKS = False  # Optional large FLAMINGO parent-cache diagnostic.

### Introduction 

This notebook studies the relationships between the kinematic properites across the TNG300 and FLAMINGO simulations which will be later used as a feature set for the Stan and NPE model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from pathlib import Path
import pickle

# FLAMINGO
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)

with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)

# TNG
tng_dir = PROJECT / "analogues/tng_for_comparison_v3"
tng_sep            = TNG_DATA["separation"]
tng_vr             = TNG_DATA["vr"]
tng_vt             = TNG_DATA["vt"]
tng_log_m200c      = TNG_DATA["log_m200c"]
tng_log_mass_ratio = TNG_DATA["log_mass_ratio"]
tng_same_host      = TNG_DATA["have_same_host"]

FLAMINGO_COLOR = "#E07B3F"
TNG_COLOR      = "#4C72B0"
features       = ["separation", "vr", "vt", "log_mass_ratio", "same_host"]

print(f"FLAMINGO pairs: {len(pairs['i']):,}")
print(f"TNG pairs:      {len(tng_sep):,}")

In [ ]:
valid_f = np.isfinite(pairs["log_tot_virial_mass"])

flam_df = pd.DataFrame({
    "host_i": sample["host_top_index"][pairs["i"]][valid_f],
    "host_j": sample["host_top_index"][pairs["j"]][valid_f],
    "separation":     pairs["separation_kpc"][valid_f],
    "vr":             pairs["vr"][valid_f],
    "vt":             pairs["vt"][valid_f],
    "log_mass_ratio": pairs["log_mass_ratio"][valid_f],
    "same_host":      pairs["have_same_host"][valid_f].astype(int),
    "log_m200c":      pairs["log_tot_virial_mass"][valid_f],
}).dropna()

valid_t = np.isfinite(tng_log_m200c)

tng_df = pd.DataFrame({
    "host_i": TNG_DATA["grnr"][TNG_DATA["pairs_i"]][valid_t],
    "host_j": TNG_DATA["grnr"][TNG_DATA["pairs_j"]][valid_t],
    "separation":     tng_sep[valid_t],
    "vr":             tng_vr[valid_t],
    "vt":             tng_vt[valid_t],
    "log_mass_ratio": tng_log_mass_ratio[valid_t],
    "same_host":      tng_same_host[valid_t].astype(int),
    "log_m200c":      tng_log_m200c[valid_t],
}).dropna()

# Subpopulation splits for EDA in coming cells
flam_same = flam_df[flam_df["same_host"] == 1]
flam_diff = flam_df[flam_df["same_host"] == 0]
tng_same  = tng_df[tng_df["same_host"] == 1]
tng_diff  = tng_df[tng_df["same_host"] == 0]

print(f"FLAMINGO: {len(flam_df):,} total  |  same-host: {len(flam_same):,}  different-host: {len(flam_diff):,}")
print(f"TNG:      {len(tng_df):,} total  |  same-host: {len(tng_same):,}  different-host: {len(tng_diff):,}")
assert len(tng_df) == TNG_MANIFEST["pair_count"], "Unexpected loss of corrected TNG rows."

In [ ]:
features = ["separation", "vr", "vt", "log_mass_ratio", "same_host"]

def print_corr_table(flam_sub, tng_sub, label):
    print(f"\n--- {label} ---")
    print(f"{'Feature':<15}  {'FLAM Pearson':>12}  {'FLAM Spearman':>13}  {'TNG Pearson':>11}  {'TNG Spearman':>12}")
    for feat in features:
        if feat == "same_host" and label != "All pairs":
            continue  # same_host is constant within subpopulations
        fp = flam_sub[feat].corr(flam_sub["log_m200c"])
        fs, _ = spearmanr(flam_sub[feat], flam_sub["log_m200c"])
        if len(tng_sub) >= 10:
            tp = tng_sub[feat].corr(tng_sub["log_m200c"])
            ts, _ = spearmanr(tng_sub[feat], tng_sub["log_m200c"])
        else:
            tp, ts = np.nan, np.nan
        print(f"{feat:<15}  {fp:>12.4f}  {fs:>13.4f}  {tp:>11.4f}  {ts:>12.4f}")

print_corr_table(flam_df,   tng_df,   "All pairs")
print_corr_table(flam_same, tng_same, "Same-host only")
print_corr_table(flam_diff, tng_diff, "Different-host only")

From the above we note that log_mass_ratio is the strongest predictor for m200c followed by vr and vt. vr and vt tend to be a better predictor in FLAMINGO but this maybe due to the limited sample size in TNG300. The informativeness of pair separation as a halo mass tracer depends on whether the pair shares a host halo and this effect is simulation-dependent.


### Mutual Info


In [ ]:
from sklearn.feature_selection import mutual_info_regression

results = []
for label, flam_sub, tng_sub in [
    ("All",          flam_df,   tng_df),
    ("Same-host",    flam_same, tng_same),
    ("Diff-host",    flam_diff, tng_diff),
]:
    feats = [f for f in features if not (f == "same_host" and label != "All")]
    
    X_f = flam_sub[feats].values
    y_f = flam_sub["log_m200c"].values
    mi_f = mutual_info_regression(X_f, y_f, random_state=42)

    if len(tng_sub) >= 10:
        X_t = tng_sub[feats].values
        y_t = tng_sub["log_m200c"].values
        mi_t = mutual_info_regression(X_t, y_t, random_state=42)
    else:
        mi_t = [np.nan] * len(feats)

    for feat, mf, mt in zip(feats, mi_f, mi_t):
        results.append({"subset": label, "feature": feat,
                        "MI_FLAMINGO": mf, "MI_TNG": mt})

mi_df = pd.DataFrame(results)
display(mi_df.pivot_table(index="feature", columns="subset",
                           values=["MI_FLAMINGO", "MI_TNG"]).round(4))

# Bar plot for all pairs only for clarity
mi_all = mi_df[mi_df["subset"] == "All"].sort_values("MI_FLAMINGO", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(mi_all))
ax.bar(x - 0.2, mi_all["MI_FLAMINGO"], 0.4, color=FLAMINGO_COLOR, label="FLAMINGO")
ax.bar(x + 0.2, mi_all["MI_TNG"],      0.4, color=TNG_COLOR,      label="TNG300-1")
ax.set_xticks(x)
ax.set_xticklabels(mi_all["feature"], rotation=20)
ax.set_ylabel("Mutual Information")
ax.set_title("Feature Informativeness for log M200c — All pairs")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_08.pdf", bbox_inches="tight")
plt.show()

> Historical interpretation from before the corrected comparison rerun. Reassess quantitative and causal claims using the new outputs.

In FLAMINGO same_host flag carries roughly 4x more information than TNG300. This is likely due to the significant overlap within the TNG300 same-host and different-host populations whereas in the FLAMINGO simulation the difference is much more prominent. This is verified quickly below:


In [ ]:
print("FLAMINGO log M200c by host type:")
print(f"  same-host:  mean={flam_same['log_m200c'].mean():.3f}  std={flam_same['log_m200c'].std():.3f}")
print(f"  diff-host:  mean={flam_diff['log_m200c'].mean():.3f}  std={flam_diff['log_m200c'].std():.3f}")

print("\nTNG log M200c by host type:")
print(f"  same-host:  mean={tng_same['log_m200c'].mean():.3f}  std={tng_same['log_m200c'].std():.3f}")
print(f"  diff-host:  mean={tng_diff['log_m200c'].mean():.3f}  std={tng_diff['log_m200c'].std():.3f}")

In [ ]:
subsets = [
    (flam_df,   tng_df,   "All pairs"),
    (flam_same, tng_same, "Same-host"),
    (flam_diff, tng_diff, "Different-host"),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("Feature Correlation Matrix by Subpopulation", fontsize=13)

for col, (flam_sub, tng_sub, label) in enumerate(subsets):
    for row, (df, sim_label, cmap) in enumerate([
        (flam_sub, f"FLAMINGO — {label}", "Oranges"),
        (tng_sub,  f"TNG300-1 — {label}", "Blues"),
    ]):
        ax = axes[row, col]
        corr = df[[c for c in df.columns if c != "same_host" or label == "All pairs"]].corr()
        im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set_xticks(range(len(corr)))
        ax.set_yticks(range(len(corr)))
        ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(corr.columns, fontsize=8)
        ax.set_title(sim_label, fontsize=9)
        for i in range(len(corr)):
            for j in range(len(corr)):
                ax.text(j, i, f"{corr.iloc[i,j]:.2f}",
                        ha="center", va="center", fontsize=7,
                        color="white" if abs(corr.iloc[i,j]) > 0.6 else "black")

plt.colorbar(im, ax=axes, shrink=0.6, label="Pearson r")
plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_11.pdf", bbox_inches="tight")
plt.show()

In [ ]:
plot_features = ["separation", "vr", "vt", "log_mass_ratio"]

subsets = [
    (flam_df,   tng_df,   "All"),
    (flam_same, tng_same, "Same-host"),
    (flam_diff, tng_diff, "Diff-host"),
]

fig, axes = plt.subplots(6, 4, figsize=(18, 24))
fig.suptitle("Feature vs log M200c — by subpopulation", fontsize=13)

for row_pair, (flam_sub, tng_sub, label) in enumerate(subsets):
    for col, feat in enumerate(plot_features):
        # FLAMINGO row
        ax = axes[row_pair * 2, col]
        ax.hexbin(flam_sub[feat], flam_sub["log_m200c"],
                  gridsize=40, cmap="Oranges", mincnt=1)
        ax.set_xlabel(feat, fontsize=8)
        ax.set_ylabel("log M200c", fontsize=8)
        ax.set_title(f"FLAMINGO — {label}", fontsize=8)

        # TNG row
        ax = axes[row_pair * 2 + 1, col]
        ax.hexbin(tng_sub[feat], tng_sub["log_m200c"],
                  gridsize=40, cmap="Blues", mincnt=1)
        ax.set_xlabel(feat, fontsize=8)
        ax.set_ylabel("log M200c", fontsize=8)
        ax.set_title(f"TNG300-1 — {label}", fontsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_12.pdf", bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

FLAMINGO_COLOR = "#E07B3F"
TNG_COLOR      = "#4C72B0"

results_rf = {}

for sim_label, df in [("FLAMINGO", flam_df), ("TNG300-1", tng_df)]:
    X = df[features].values
    y = df["log_m200c"].values

    train_idx, test_idx = group_split(df, test_size=0.2, seed=42)
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_train, y_train)

    r2_train = rf.score(X_train, y_train)
    r2_test  = rf.score(X_test,  y_test)

    perm = permutation_importance(
        rf, X_test, y_test,
        n_repeats=30,
        random_state=42,
        n_jobs=-1
    )

    results_rf[sim_label] = {
        "rf":          rf,
        "r2_train":    r2_train,
        "r2_test":     r2_test,
        "perm_mean":   perm.importances_mean,
        "perm_std":    perm.importances_std,
    }

    print(f"\n{sim_label}")
    print(f"  R² train: {r2_train:.4f}  |  R² test: {r2_test:.4f}")
    print(f"  Permutation importance:")
    order = np.argsort(perm.importances_mean)[::-1]
    for idx in order:
        print(f"    {features[idx]:<15}  {perm.importances_mean[idx]:>8.4f} ± {perm.importances_std[idx]:.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Random Forest Permutation Importance for log M200c", fontsize=13)

for ax, (sim_label, res), color in zip(
    axes,
    results_rf.items(),
    [FLAMINGO_COLOR, TNG_COLOR]
):
    order = np.argsort(res["perm_mean"])[::-1]
    feat_ordered  = [features[i] for i in order]
    means_ordered = res["perm_mean"][order]
    stds_ordered  = res["perm_std"][order]

    ax.barh(feat_ordered[::-1], means_ordered[::-1],
            xerr=stds_ordered[::-1],
            color=color, alpha=0.8, capsize=4)
    ax.set_xlabel("Mean decrease in R²", fontsize=10)
    ax.set_title(f"{sim_label}  (R²={res['r2_test']:.3f})", fontsize=11)
    ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "comparison_13.pdf", bbox_inches="tight")
plt.show()

This notebook discusses reliability of log_mass_ratio, vr, vt, seperation and same_host as some of the features. In the next notebook, stellar mass, photometry, velocity dispersion, morphology and metallicity consistency are assessed.
